In [ ]:
# ── Cell 0 ── config & imports
import json, gzip, io
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd

LOG_PATH = Path("~/.sigwood/exports/cloudtrail/cloudtrail_20260520_to_20260603_00h.json.log")  # <- point at the big file

# Curated projection for the Drain3 path (cell 2+). Not used yet.
PROJECTION_FIELDS = [
    "eventSource", "eventName", "userIdentity.type",
    "principal", "sourceIPAddress", "errorCode",
]

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

In [ ]:
# ── Cell 1 ── load one file → flat list of event dicts, then summarize
def load_cloudtrail(path: Path) -> list[dict]:
    raw = path.read_bytes()
    # sniff gzip (magic bytes 1f 8b)
    if raw[:2] == b"\x1f\x8b":
        raw = gzip.decompress(raw)
    text = raw.decode("utf-8", errors="replace").strip()

    events, bad = [], 0
    # Try whole-file JSON first (covers {"Records":[...]} and a bare [...] array)
    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and "Records" in obj:
            events = obj["Records"]
        elif isinstance(obj, list):
            events = obj
        else:
            events = [obj]
    except json.JSONDecodeError:
        # Fall back to JSONL (one object per line)
        for line in text.splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                events.append(json.loads(line))
            except json.JSONDecodeError:
                bad += 1
    if bad:
        print(f"  ! skipped {bad} unparseable line(s)")
    return events

events = load_cloudtrail(LOG_PATH)

# ── shape summary ──
print(f"events loaded: {len(events)}")
if events:
    times = sorted(e.get("eventTime", "") for e in events if e.get("eventTime"))
    print(f"time span:     {times[0]}  ->  {times[-1]}")

    keycount = Counter(k for e in events for k in e)
    print(f"\ntop-level keys (presence of {len(events)}):")
    for k, v in keycount.most_common():
        print(f"  {v:6d}  {k}")

    print("\neventSource (top 15):")
    for k, v in Counter(e.get("eventSource") for e in events).most_common(15):
        print(f"  {v:6d}  {k}")

    print("\nreadOnly / eventType / userIdentity.type:")
    print("  readOnly:  ", dict(Counter(e.get("readOnly") for e in events)))
    print("  eventType: ", dict(Counter(e.get("eventType") for e in events)))
    print("  idType:    ", dict(Counter(e.get("userIdentity", {}).get("type") for e in events)))

    errs = Counter(e.get("errorCode") for e in events if e.get("errorCode"))
    print(f"\nerrorCode (events with one): {sum(errs.values())}")
    for k, v in errs.most_common(10):
        print(f"  {v:6d}  {k}")
        

In [ ]:
# ── Cell 2 ── eyeball the rare / interesting tail before modeling
def principal(e):
    p = e.get("userIdentity", {}).get("principalId", "") or ""
    return p.split(":")[-1] if ":" in p else (p or e.get("userIdentity", {}).get("type", "?"))

def brief(e):
    return {
        "time": e.get("eventTime"),
        "src": e.get("eventSource", "").replace(".amazonaws.com", ""),
        "event": e.get("eventName"),
        "principal": principal(e),
        "ip": e.get("sourceIPAddress"),
        "err": e.get("errorCode"),
    }

print("═══ WRITES (readOnly == False) ═══")
for e in sorted([x for x in events if x.get("readOnly") is False], key=lambda x: x.get("eventTime","")):
    b = brief(e); print(f"  {b['time']}  {b['src']:18s} {b['event']:34s} {b['principal']:16s} {b['ip']}")

print("\n═══ ERRORS ═══")
for e in sorted([x for x in events if x.get("errorCode")], key=lambda x: x.get("eventTime","")):
    b = brief(e); print(f"  {b['time']}  {b['err']:30s} {b['src']:14s} {b['event']:28s} {b['principal']}")

print("\n═══ CONSOLE SIGN-INS & non-AssumedRole identities ═══")
for e in sorted([x for x in events if x.get("eventType")=="AwsConsoleSignIn"
                 or x.get("userIdentity",{}).get("type") in ("IdentityCenterUser","SAMLUser","Root")],
                key=lambda x: x.get("eventTime","")):
    b = brief(e); print(f"  {b['time']}  {e.get('eventType'):18s} {b['event']:22s} {b['principal']:16s} {b['ip']}")

print("\n═══ AwsServiceEvent ═══")
for e in sorted([x for x in events if x.get("eventType")=="AwsServiceEvent"], key=lambda x: x.get("eventTime","")):
    b = brief(e); print(f"  {b['time']}  {b['src']:18s} {b['event']:34s}")

# distinct human actors & IPs, to confirm the "one human" assumption holds at scale
humans = [e for e in events if e.get("userIdentity",{}).get("type") in ("AssumedRole","IdentityCenterUser","SAMLUser","Root")]
print(f"\ndistinct principals (human-ish): {dict(Counter(principal(e) for e in humans).most_common(10))}")
print(f"distinct source IPs (non-AWS):  {dict(Counter(e.get('sourceIPAddress') for e in humans if not str(e.get('sourceIPAddress','')).endswith('amazonaws.com')).most_common(10))}")

In [ ]:
# ── Cell 3 ── unmask resource-explorer-2: full identity context
import json
from collections import Counter

def principal(e):
    p = e.get("userIdentity", {}).get("principalId", "") or ""
    return p.split(":")[-1] if ":" in p else (p or e.get("userIdentity", {}).get("type", "?"))

crawler = [e for e in events if principal(e) == "resource-explorer-2"]
print(f"resource-explorer-2 events: {len(crawler)}\n")

# What does its userIdentity actually look like? Sample one in full.
print("═══ sample full userIdentity ═══")
print(json.dumps(crawler[0].get("userIdentity", {}), indent=2))

# invokedBy is the tell for service-linked crawling
print("\n═══ userIdentity.invokedBy ═══")
print(dict(Counter(e.get("userIdentity", {}).get("invokedBy") for e in crawler)))

print("\n═══ userIdentity.type ═══")
print(dict(Counter(e.get("userIdentity", {}).get("type") for e in crawler)))

# distinct role ARNs / principalIds behind it
print("\n═══ distinct arn ═══")
for k, v in Counter(e.get("userIdentity", {}).get("arn") for e in crawler).most_common():
    print(f"  {v:5d}  {k}")

print("\n═══ distinct sessionContext.sessionIssuer.arn ═══")
for k, v in Counter(e.get("userIdentity", {}).get("sessionContext", {}).get("sessionIssuer", {}).get("arn")
                    for e in crawler).most_common():
    print(f"  {v:5d}  {k}")

# what services does it sweep, and when - does cadence look like scheduled automation?
print("\n═══ top services swept ═══")
for k, v in Counter(e.get("eventSource", "").replace(".amazonaws.com", "") for e in crawler).most_common(15):
    print(f"  {v:5d}  {k}")

print("\n═══ events per day (cadence) ═══")
for k, v in sorted(Counter(e.get("eventTime", "")[:10] for e in crawler).items()):
    print(f"  {k}  {v:5d}")

In [ ]:
# ── Cell 4 ── assign lanes, confirm split, build Lane A feature matrix
import numpy as np
import pandas as pd
from collections import Counter

KNOWN_HUMAN_IPS = {"1.1.1.1"}  # your real source IP from cells 2-3

def principal(e):
    p = e.get("userIdentity", {}).get("principalId", "") or ""
    return p.split(":")[-1] if ":" in p else (p or e.get("userIdentity", {}).get("type", "?"))

def lane(e):
    ui = e.get("userIdentity", {})
    t = ui.get("type")
    invoked = ui.get("invokedBy", "")
    # Lane B: Resource Explorer service-linked role (fully characterized in cell 3)
    if principal(e) == "resource-explorer-2" or "AWSServiceRoleForResourceExplorer" in (ui.get("arn") or ""):
        return "B_resexplorer"
    # Lane A: interactive human surface - user's sessions + the auth/signin events
    if e.get("eventType") in ("AwsConsoleSignIn",) or t in ("IdentityCenterUser", "SAMLUser", "Root"):
        return "A_human"
    if t == "AssumedRole" and principal(e) == "user":
        return "A_human"
    # Lane C: everything else AWS-side (service principals, other SLRs, background)
    if t == "AWSService" or invoked.endswith("amazonaws.com"):
        return "C_awsservice"
    return "C_awsservice"

for e in events:
    e["_lane"] = lane(e)

print("═══ lane breakdown ═══")
for k, v in Counter(e["_lane"] for e in events).most_common():
    print(f"  {v:5d}  {k}")

# sanity: who's in Lane A, and from where
laneA = [e for e in events if e["_lane"] == "A_human"]
print(f"\nLane A principals: {dict(Counter(principal(e) for e in laneA).most_common())}")
print(f"Lane A source IPs: {dict(Counter(e.get('sourceIPAddress') for e in laneA).most_common())}")
print(f"Lane A eventSources: {dict(Counter(e.get('eventSource','').replace('.amazonaws.com','') for e in laneA).most_common(12))}")

# ── feature matrix on Lane A ──
def hour(e):
    t = e.get("eventTime", "")
    return int(t[11:13]) if len(t) >= 13 else -1

rows = []
for e in laneA:
    ui = e.get("userIdentity", {})
    rows.append({
        "eventTime": e.get("eventTime"),
        "eventSource": e.get("eventSource", "").replace(".amazonaws.com", ""),
        "eventName": e.get("eventName"),
        "principal": principal(e),
        "ip_known": e.get("sourceIPAddress") in KNOWN_HUMAN_IPS,
        "hour": hour(e),
        "readOnly": bool(e.get("readOnly")),
        "has_error": bool(e.get("errorCode")),
        "from_console": e.get("sessionCredentialFromConsole") == "true",
        "has_resources": bool(e.get("resources")),
        "has_tls": bool(e.get("tlsDetails")),
        "mfa": e.get("userIdentity", {}).get("sessionContext", {}).get("attributes", {}).get("mfaAuthenticated") == "true",
        "id_type": ui.get("type"),
    })
df = pd.DataFrame(rows)

# encode: one-hot the categoricals, keep booleans as 0/1, hour as cyclic
cat_cols = ["eventSource", "eventName", "id_type"]
bool_cols = ["ip_known", "readOnly", "has_error", "from_console", "has_resources", "has_tls", "mfa"]

X = pd.get_dummies(df[cat_cols], prefix=cat_cols)
for c in bool_cols:
    X[c] = df[c].astype(int)
# cyclic hour so 23:00 and 00:00 are near each other
X["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
X["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

print(f"\n═══ Lane A feature matrix ═══")
print(f"shape: {X.shape}  ({X.shape[0]} events × {X.shape[1]} features)")
print(f"\nfeature columns ({X.shape[1]}):")
print("  " + "\n  ".join(X.columns.tolist()))
print(f"\nfirst 6 rows (subset of cols):")
show = [c for c in X.columns if c.startswith(("eventName_","ip_known","has_error","from_console"))][:8]
print(df[["eventTime","eventName"]].head(6).to_string(index=False))
print(X[show].head(6).to_string(index=False))

In [ ]:
# ── Cell 5 ── bake-off: Isolation Forest vs. rarity ranking on Lane A
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

# ---------- Approach 1: Isolation Forest on the raw feature matrix ----------
iso = IsolationForest(n_estimators=300, contamination="auto", random_state=42)
iso.fit(X)
df["if_score"] = iso.decision_function(X)      # higher = more normal
df["if_outlier"] = iso.predict(X) == -1        # True = flagged anomaly

n_flag = int(df["if_outlier"].sum())
print(f"═══ APPROACH 1: Isolation Forest ═══")
print(f"flagged {n_flag} / {len(df)} events as anomalous\n")
print("most-anomalous 15 (lowest score):")
top_if = df.sort_values("if_score").head(15)
print(top_if[["eventTime","eventSource","eventName","has_error","ip_known"]].to_string(index=False))

print("\nwhat eventNames dominate the flagged set:")
print(df[df["if_outlier"]]["eventName"].value_counts().head(12).to_string())

# ---------- Approach 2: rarity ranking (how novel is this action for this principal?) ----------
print(f"\n\n═══ APPROACH 2: rarity ranking - eventName frequency over 14 days ═══")
freq = df["eventName"].value_counts()
df["name_count"] = df["eventName"].map(freq)

# the rare tail: actions seen only a handful of times in the whole window
rare = (freq[freq <= 3]
        .sort_values()
        .rename_axis("eventName")
        .reset_index(name="times_seen"))
print(f"{len(rare)} distinct actions seen ≤3× in 14 days (the 'rare tail'):\n")
# attach the eventSource + when-first-seen for context
first_seen = df.sort_values("eventTime").groupby("eventName").first()
rare["eventSource"] = rare["eventName"].map(first_seen["eventSource"])
rare["first_seen"]  = rare["eventName"].map(first_seen["eventTime"])
rare["had_error"]   = rare["eventName"].map(df.groupby("eventName")["has_error"].any())
print(rare.to_string(index=False))

# ---------- Overlap: do the two approaches agree? ----------
print(f"\n\n═══ AGREEMENT ═══")
if_flagged_names = set(df[df["if_outlier"]]["eventName"])
rare_names = set(rare["eventName"])
print(f"actions IF flagged but rarity did NOT: {sorted(if_flagged_names - rare_names)}")
print(f"actions rarity flagged but IF did NOT: {sorted(rare_names - if_flagged_names)}")
print(f"both agree on:                         {sorted(if_flagged_names & rare_names)}")

# ---------- What's the dominant-class problem, quantified ----------
print(f"\n═══ context: how dominated is Lane A? ═══")
print(df["eventName"].value_counts().head(5).to_string())
print(f"\ntop action = {freq.iloc[0]} / {len(df)} = {100*freq.iloc[0]/len(df):.1f}% of Lane A")